In [5]:
from datetime import time

import dilutionUtils as dil

In [ ]:
files, injection, calibration = dil.load_config("./outputs/202504_trial1_solution_config.ini")

In [2]:
csv_files ="."
search_string = "output/20250709_*_790478.csv"
calib_file = './calibration.csv'
cal_data = dil.read_csv(calib_file)


In [3]:
x = dil.extract_fields(cal_data)

In [4]:
# This is important
# Mixing the Primary sample
# note that on 07/09 the injection sample used to calibrate was taken from the second injection but used to calibrate both trials.
Vo = 12000 # mL of H20 collected stream
rho_salt = 2.165 # density of NaCl in g/cm3
mass_salt = 4000 # g
Vc = x['Vol_secondary_mL'][0]
vol_calibration_sample = 100 # ml
Vnew = Vo +  mass_salt / rho_salt
vol_solution_injected = Vnew - vol_calibration_sample # ml
injection_conc = (mass_salt) / (Vo) # g/ml

print(injection_conc, Vnew)

0.3333333333333333 13847.57505773672


In [5]:
RC_values = dil.calibration_RC(injection_conc, x['Vol_cal_solution_mL'], Vc)

In [6]:
# returns np.float64(5.728194064066105e-07) with the given calibration file
# is that right?
slope_k = dil.calculate_k(RC_values, x['actual_CD_uS_per_cm'])

In [7]:
data = dil.get_file_paths(csv_files, search_string)
data

['./output/20250709_144014_790478.csv',
 './output/20250709_150524_790478.csv',
 './output/20250709_160924_790478.csv']

In [8]:
ec_data_trial_1 = dil.extract_fields(dil.read_csv(data[0]))
ec_data_trial_2 = dil.extract_fields(dil.read_csv(data[1]))

In [ ]:
# better that get_measurement_values function
def time_diff_sec(start:str, end:str):
    """ Calculate the diffence between to times in seconds.
    
        Parameters:
            start (string): "H:M:S"
            end (string): "H:M:S"
        Returns:
            float: difference in seconds between the inputs.
        """
    t1 = datetime.strptime(start,"%H:%M:%S")
    t2 = datetime.strptime(end,"%H:%M:%S")
    diff_seconds = (t2-t1).total_seconds()
    
    return int(diff_seconds)

'\nmsnt1_start = time(14,50,33) #"14:50:33"\nmsnt1_end = time(14,58,6) #"14:58:06"\nstart_time = datetime.strptime(msnt1_start, "%H:%M:%S").time()\nend_time = datetime.strptime(msnt1_end, "%H:%M:%S").time()\nprint(type(start_time), type(end_time))'

In [10]:
msnt1_start = time(14,50,33) #"14:50:33"
msnt1_end = time(14,58,6) #"14:58:06"
# Pylance prefers it like this
result1 = dil.get_measurement_time_values(msnt1_start, msnt1_end, ec_data_trial_1, show_data=True)
if result1 is not None:
    background1, trial1, trial_time1 = result1
else:
    print("Start or end time not found in ec_data_trial_1['Date Time']")

In [11]:
time_list1 = trial_time1
time_diffs1, time_elapsed1 = dil.compute_time_differences_in_seconds(time_list1)
EC_bg1 = dil.get_background_ec(ec_data=background1,start_time=None,end_time=None)
Q_1 = dil.calculate_Q(EC_bg1,trial1,slope_k,time_diffs1,vol_solution_injected)

In [12]:
Q_1

np.float64(31.62416055525219)

In [13]:

msnt2_start = time(15,17,38) #"15:17:38"
msnt2_end = time(15,24,41) #"15:24:41"
# Pylance prefers it like this
result2 = dil.get_measurement_time_values(msnt2_start, msnt2_end, ec_data_trial_2, show_data=True)
if result2 is not None:
    background_2, trial_2, trial_time_2 = result2
else:
    print("Start or end time not found in ec_data_trial_2['Date Time']")

In [14]:
time_list2 = trial_time_2
time_diffs2, time_elapsed2 = dil.compute_time_differences_in_seconds(time_list2)
EC_bg2 = dil.get_background_ec(ec_data=background_2,start_time=None,end_time=None)
Q_2 = dil.calculate_Q(EC_bg2,trial_2,slope_k,time_diffs2,vol_solution_injected)

In [15]:
Q_2

np.float64(32.998944252434434)